# U-Net Segmentation Model Training

This notebook trains a U-Net model for diabetic ulcer boundary segmentation.

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging

print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

## Load Training Data

In [ ]:
class UlcerSegmentationDataset(Dataset):
    """Dataset for ulcer segmentation with images and masks"""
    
    def __init__(self, image_dir: str, mask_dir: str):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.images = list(self.image_dir.glob("*.jpg")) + list(self.image_dir.glob("*.png"))
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Placeholder for actual image loading
        image = torch.randn(3, 256, 256)
        mask = torch.randint(0, 2, (1, 256, 256)).float()
        return image, mask

# Initialize dataset
dataset = UlcerSegmentationDataset("datasets/images/ulcers", "datasets/segmentation_masks")
print(f"Dataset size: {len(dataset)}")

## Define U-Net Architecture

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        
        # Encoder
        self.enc1 = self._conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.enc2 = self._conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.enc3 = self._conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # Bottleneck
        self.bottleneck = self._conv_block(256, 512)
        
        # Decoder
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = self._conv_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = self._conv_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = self._conv_block(128, 64)
        
        self.final = nn.Conv2d(64, out_channels, 1)
    
    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        enc1 = self.enc1(x)
        x = self.pool1(enc1)
        enc2 = self.enc2(x)
        x = self.pool2(enc2)
        enc3 = self.enc3(x)
        x = self.pool3(enc3)
        x = self.bottleneck(x)
        x = self.upconv3(x)
        x = torch.cat([x, enc3], 1)
        x = self.dec3(x)
        x = self.upconv2(x)
        x = torch.cat([x, enc2], 1)
        x = self.dec2(x)
        x = self.upconv1(x)
        x = torch.cat([x, enc1], 1)
        x = self.dec1(x)
        x = self.final(x)
        return torch.sigmoid(x)

model = UNet(3, 1)
print(model)

## Training Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Training on: {device}")

## Train Model

In [ ]:
epochs = 5
losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for batch_idx, (images, masks) in enumerate(dataloader):
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

print("Training completed!")

## Plot Training Results

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('U-Net Segmentation Model Training Loss')
plt.grid(True)
plt.show()

## Save Model

In [ ]:
Path("model_weights").mkdir(exist_ok=True)
torch.save(model.state_dict(), "model_weights/segmentation_model.pth")
print("Model saved to model_weights/segmentation_model.pth")